# Workshop: Building Gemma 3 from Scratch
## Notebook 6: RMSNorm and Double Normalization

**Estimated Time: 10 minutes**

Modern Gemma models use **Root Mean Square Layer Normalization (RMSNorm)**. It is simpler and faster than standard LayerNorm because it skips the mean calculation — only computing RMS (root mean square).

## Learning Objectives:
1. Understand the math of RMSNorm.
2. Implement the Gemma-specific "Add-One" RMSNorm layer.
3. Learn about the **"double-norm"** (Pre + Post) for every sub-layer — Gemma's signature trick.

In [1]:
import torch
import torch.nn as nn
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

dim = 768  # Gemma 3: hidden_size
eps = 1e-6

Python version: 3.12.13 (main, Mar 10 2026, 18:15:41) [Clang 21.1.4 ]
PyTorch version: 2.5.1


## 1. RMSNorm Math

## LayerNorm vs RMSNorm

**LayerNorm**: $y = \frac{x - E[x]}{\sqrt{Var[x] + \epsilon}} \cdot \gamma + \beta$

**RMSNorm**: $y = \frac{x}{\sqrt{Mean(x^2) + \epsilon}} \cdot \gamma$

RMSNorm skips the mean subtraction ($E[x]$), which is a simplification that works well in practice. The weight $\gamma$ is a learned parameter vector that scales the normalized output.

## The Gemma "Add-One" Trick

Gemma's key innovation: the weight is initialized to **zero** and the formula uses **$(1 + \gamma)$** as the scaling factor.

At training start, $\gamma = 0$, so $(1 + \gamma) = 1$, and the layer is a **pure identity** (just RMS-normalize, scale by 1). This is far more stable than initializing $\gamma = 1$ and using it directly.

In [2]:
class RMSNorm(nn.Module):
    """Gemma 3-style RMSNorm with Add-One initialization."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Gemma's Add-One trick: weight starts at zero
        self.weight = nn.Parameter(torch.zeros(dim))

    def _norm(self, x):
        # x: (... , dim)
        # RMS = sqrt(mean(x^2))
        # return x / RMS
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        # Apply (1 + weight) scaling — at init this is just 1.0 (identity)
        return self._norm(x.float()).type_as(x) * (1.0 + self.weight)

norm = RMSNorm(dim)
x = torch.randn(2, 5, dim) * 10  # High variance input
out = norm(x)

print(f"Input variance:  {x.var().item():.4f}")
print(f"Output variance: {out.var().item():.4f}")

# At initialization, weight=0 so output is just normalized (variance ~1)
print(f"Weight: {norm.weight.data[:3].tolist()}... (all zeros)")
print(f"Effective scale: {(1 + norm.weight.data[0]).item():.4f}")
assert out.shape == x.shape
print("✅ RMSNorm works correctly!")

Input variance:  100.7250
Output variance: 0.9996
Weight: [0.0, 0.0, 0.0]... (all zeros)
Effective scale: 1.0000
✅ RMSNorm works correctly!


## 2. The "Double-Norm" Secret: Pre-Norm + Post-Norm on EVERY sub-layer

Gemma 3/3 is unique: **every** sub-layer gets **both** pre-norm and post-norm. This is "double normalization":

```python
# For the Attention sub-layer:
x = input 
x = RMSNorm_pre(x)           # normalize before
x = Attention(x)             # apply
x = x + RMSNorm_post(x)     # normalize + residual  <-- unique!

# For the MLP sub-layer:
x = RMSNorm_pre(x)           # normalize before
x = MLP(x)                   # apply
x = x + RMSNorm_post(x)     # normalize + residual  <-- unique!
```

## 3. Why Double-Norm Works So Well

Comparing architectures:

| Architecture | Norm locations | Stability |
|---|---|---|
| **Post-norm** (original Transformer) | After each sub-layer only | Unstable at depth |
| **Pre-norm** (T5, Llama) | Before each sub-layer only | Moderate |
| **Double-norm** (Gemma 3/3) | Pre + Post on **every** sub-layer | **Most stable** |

The post-norm is the innovation that sets Gemma apart from Llama.

## Exercise:
Why does initializing the weight to zero and using $(1 + weight)$ help with training stability compared to initializing the weight to one and using it directly?

In [3]:
# Answer: The model starts in identity mode
init_weight = torch.zeros(dim)
init_scale = 1.0 + init_weight[0]  # = 1.0
print(f"At initialization: scale = {init_scale:.1f} (exactly 1.0)")

# The layer is essentially: RMSNormalize(x) * 1.0 = RMSNormalize(x)
# The signal flows through untouched — gradients are not skewed by random scaling initially.
print("\nThe layer acts as RMS-normalization-only at the start.")
print("Gradients can propagate freely before norm parameters diverge from zero.")

At initialization: scale = 1.0 (exactly 1.0)

The layer acts as RMS-normalization-only at the start.
Gradients can propagate freely before norm parameters diverge from zero.


## Key Takeaway

Gemma 3's signature normalization: **Double RMSNorm** (Pre + Post on every sub-layer) with **Add-One** weight initialization. This makes Gemma 3 incredibly stable even with deep architectures.